# Floyd–Warshall on CPU and GPU (Numba CUDA)

This notebook provides:

- A NumPy baseline implementation of Floyd–Warshall (all-pairs shortest paths)
- A Numba CUDA kernel that mirrors the classic per-`k` GPU update
- A small correctness check on a toy graph
- Simple timing comparison on a random dense graph (if a CUDA GPU is available)


In [ ]:
import numpy as np
import math, time
try:
    from numba import cuda
    has_cuda = cuda.is_available()
except Exception:
    has_cuda = False
print("CUDA available:", has_cuda)


## NumPy baseline implementation

We use the standard triple-loop Floyd–Warshall algorithm on a dense adjacency matrix.


In [ ]:
INF = 1e20

def floyd_warshall_numpy(W: np.ndarray) -> np.ndarray:
    """In-place Floyd–Warshall on a dense NumPy array.
    W[i, j] is the initial distance (INF if no edge)."""
    D = W.copy().astype(np.float32)
    n = D.shape[0]
    for k in range(n):
        Dik = D[:, k][:, None]     # shape (n,1)
        Dkj = D[k, :][None, :]     # shape (1,n)
        D = np.minimum(D, Dik + Dkj)
    return D


## Numba CUDA kernel

We mirror the CUDA C++ kernel: for each fixed `k`, launch a 2D grid of threads
and let each thread update one `(i,j)` entry.


In [ ]:
if has_cuda:
    from numba import cuda, float32

    @cuda.jit
    def floyd_step_kernel(D, n, k):
        i, j = cuda.grid(2)
        if i < n and j < n:
            Dik = D[i, k]
            Dkj = D[k, j]
            alt = Dik + Dkj
            if alt < D[i, j]:
                D[i, j] = alt

    def floyd_warshall_numba_cuda(W: np.ndarray) -> np.ndarray:
        D_host = W.astype(np.float32)
        n = D_host.shape[0]
        D_dev = cuda.to_device(D_host)
        threads = (16, 16)
        blocks = ((n + threads[0] - 1) // threads[0],
                  (n + threads[1] - 1) // threads[1])
        for k in range(n):
            floyd_step_kernel[blocks, threads](D_dev, n, k)
        cuda.synchronize()
        return D_dev.copy_to_host()
else:
    def floyd_warshall_numba_cuda(W: np.ndarray) -> np.ndarray:
        raise RuntimeError("CUDA is not available in this environment.")


## Toy example for correctness

We test on a small 4-node graph where we know the expected answer.


In [ ]:
W = np.array([
    [0, 3, INF, 7],
    [8, 0, 2, INF],
    [5, INF, 0, 1],
    [2, INF, INF, 0]
], dtype=np.float32)

D_cpu = floyd_warshall_numpy(W)
print("CPU result:\n", D_cpu)

if has_cuda:
    D_gpu = floyd_warshall_numba_cuda(W)
    print("GPU result:\n", D_gpu)
    diff = np.max(np.abs(D_cpu - D_gpu))
    print("Max abs diff CPU vs GPU:", diff)
else:
    print("CUDA not available; skipping GPU test.")


## Simple timing comparison

We generate a random dense graph and compare NumPy vs. Numba CUDA timings.


In [ ]:
def random_graph(n, edge_prob=0.6, max_w=10.0, seed=42):
    rng = np.random.default_rng(seed)
    W = np.full((n, n), INF, dtype=np.float32)
    for i in range(n):
        W[i, i] = 0.0
        for j in range(n):
            if i == j:
                continue
            if rng.random() < edge_prob:
                W[i, j] = rng.uniform(1.0, max_w)
    return W

n = 256  # adjust up if you have a strong GPU
W_rand = random_graph(n)

t0 = time.time()
D_cpu = floyd_warshall_numpy(W_rand)
t1 = time.time()
print(f"NumPy (CPU) time for n={n}: {(t1 - t0)*1000:.1f} ms")

if has_cuda:
    t2 = time.time()
    D_gpu = floyd_warshall_numba_cuda(W_rand)
    t3 = time.time()
    print(f"Numba CUDA time for n={n}: {(t3 - t2)*1000:.1f} ms")
    diff = np.max(np.abs(D_cpu - D_gpu))
    print("Max abs diff CPU vs GPU:", diff)
else:
    print("CUDA not available; skipping GPU timing.")
